In [2]:
import importlib
import lung_helpers

importlib.reload(lung_helpers)
from lung_helpers import *

In [1]:
# After you download the datasets, set this to point to their parent directory
BASE_DB_DIR = './datasets/'

In [3]:
df_cohort = pd.read_csv(f"{BASE_DB_DIR}/FINAL_COHORT_LISTING.csv").set_index('main_index')

df_cohort_disc = df_cohort[df_cohort['cohort']=='discovery']
df_cohort_rad  = df_cohort[df_cohort['cohort']=='rad_valid']
df_cohort_path = df_cohort[df_cohort['cohort']=='path_valid']


In [4]:
df_clinical_old = pd.read_csv("/gpfs/mskmindhdp_emc/user/shared_data_folder/lung-mind-project/cohorts/MIND Cohort_v2_8August2020.csv")

In [29]:
df_clinical = get_clinical_table_v2(
    path=f"{BASE_DB_DIR}/18193MSKMINDProjectM-OmnibusInventory_DATA_2021-12-20_1540-WITH-TB-and-SCANNER.csv", 
    main_index_col='dmp_pt_id',cohort=df_cohort_disc)
df_clinical_rad_valid  = get_clinical_table_v2(
    path=f"{BASE_DB_DIR}/18193MSKMINDProjectM-OmnibusInventory_DATA_2021-12-20_1540-WITH-TB-and-SCANNER.csv", 
    main_index_col='did_acc',cohort=df_cohort_rad)
df_clinical_path_valid = get_clinical_table_v2(
    path=f"{BASE_DB_DIR}/18193MSKMINDProjectM-OmnibusInventory_DATA_2021-12-20_1540-WITH-TB-and-SCANNER.csv", 
    main_index_col='pdl1_image_id',cohort=df_cohort_path)

df_outcomes_PRESHUFFLE = df_clinical[['label']].copy(deep=True)


247 62
50 11
71 21


In [6]:
clinical_predictors = ['age', 
     'pack_years', 
     'ecog', 
     'albumin',
     'dnlr', 
     'brain_mets', 
     'liver_mets', 
     'tumor_burden', 
     'therapy_line', 
     'recieves_combo_therapy', 
     'site_lung', 
     'recieves_pdl1_therapy',
     'hist_adeno']

In [7]:
df_ids = get_id_table()
df_pdl1 = get_pdl1_table(df_ids)
df_genomic = get_genomic_table()
df_tmb     = df_genomic[['TMB']]
df_nontmb  = df_genomic.loc[:, ~df_genomic.columns.str.contains("TMB")]
df_texture = get_textures_table(table_names=['NEW_LUNG_glcm_Autocorrelation_v2_20x_stain1_PDL1', 'NEW_LUNG_PixelIntensity_v2_20x_stain1_PDL1'])
df_auto    = df_texture.loc[:, df_texture.columns.str.contains("Autocorrelation")]
df_skew    = df_texture.loc[:, df_texture.columns.str.contains("skewness")]
df_radiology = get_radiology_table(table_name='LUNG_RADIOMICS_spacing1.0_MirpOn_Window1350.250_allImageTypes_bw20')
df_radiology_by_site = decorate_with_site_index(df_radiology)
df_labs = df_clinical[clinical_predictors]

df_radiology_valid = get_radiology_table(table_name='LUNG_RADIOMICS_spacing1.0_MirpOn_Window1350.250_allImageTypes_bw20_VALIDATION')
df_radiology_valid_by_site = decorate_with_site_index(df_radiology_valid)

df_texture_valid = get_textures_table(table_names=['NEW_LUNG_glcm_Autocorrelation_v2_20x_stain1_PDL1_VALIDATION', 'NEW_LUNG_PixelIntensity_v2_20x_stain1_PDL1_VALIDATION']).drop(index=['1085806'])

247
201
247
105
3996
1176
53


In [8]:
df_glcm = pd.read_parquet("/gpfs/mskmind_emc/data_lake/staging/waystation/datasets/LUNG_PATHOLOGY_PDL1_GLCM_V3/data.parquet") \
    .drop(columns=['last_updated']).dropna()

df_glcm = df_glcm.set_index('slide_id').join(df_clinical.reset_index().set_index('slide_id')['main_index']).dropna().set_index('main_index')
df_glcm = df_glcm.loc[df_texture.index, ~df_glcm.columns.str.contains('_nobs|_min|_max')] 


In [9]:
modality_MASK = df_clinical[[]]
modality_dict = {}

modality_PC_full = prepare_rad_modality_by_size(modality_dict, df_radiology_by_site, modality_MASK, 'PC', 'rad_lesion_pc')
modality_PL_full = prepare_rad_modality_by_size(modality_dict, df_radiology_by_site, modality_MASK, 'PL', 'rad_lesion_pl')
modality_LN_full = prepare_rad_modality_by_size(modality_dict, df_radiology_by_site, modality_MASK, 'LN', 'rad_lesion_ln')

modality_LU_full = prepare_rad_modality_by_size(
    modality_dict, 
    df_radiology_by_site, 
    modality_MASK, 
    sites=['PC', 'PL', 'LN'], 
    name='rad_lesion_lu',
    sort='original_shape_MeshVolume', 
    ascending=False,
    reduce=True
)

prepare_other_modalities(modality_dict, df_texture, modality_MASK, 'path_ihc_pdl1')
prepare_other_modalities(modality_dict, df_skew,    modality_MASK, 'path_ihc_skew')
prepare_other_modalities(modality_dict, df_glcm,    modality_MASK, 'path_ihc_glcm')
prepare_other_modalities(modality_dict, df_genomic, modality_MASK, 'gen_driver_mut_amp')
prepare_other_modalities(modality_dict, df_nontmb,  modality_MASK, 'gen_driver_non_tmb')
prepare_other_modalities(modality_dict, df_tmb,     modality_MASK, 'gen_driver_tmb')
prepare_other_modalities(modality_dict, df_pdl1,    modality_MASK, 'cnl_pdl1_score')
prepare_other_modalities(modality_dict, df_labs,    modality_MASK, 'cnl_dem_labs')

modality_dict['cnl_pdl1_score'] = - modality_dict['cnl_pdl1_score'] / 100.0

modality_MASK = modality_MASK.fillna(False)

pd.DataFrame(modality_MASK.sum(), columns=['Count (Modality)'])

(array([1.]), array([163]))
(array([3.]), array([21]))
(array([5.]), array([67]))
(array([1., 2., 3., 4., 5., 6.]), array([133,   6,   4,   5,  28,  11]))


,Count (Modality)
rad_lesion_pc,163
rad_lesion_pl,21
rad_lesion_ln,67
rad_lesion_lu,187
path_ihc_pdl1,105
path_ihc_skew,105
path_ihc_glcm,105
gen_driver_mut_amp,247
gen_driver_non_tmb,247
gen_driver_tmb,247


In [10]:
import warnings
warnings.filterwarnings("ignore")

In [11]:
model_params = {'epochs':125, 'lr':0.01, 'alpha':0.001, 'beta':0.0, 'cross_modality_enabled':False}
model_params_gate_off = {'epochs':125, 'lr':0.01, 'alpha':0.001, 'beta':0.0, 'cross_modality_enabled':False, 'attention_gate_enabled':False}

dfs_rad_filters_lu = {
    0: {'l1_selection_df': modality_PC_full, 'kwargs':{'l1_strength': 0.1}},
    1: {'l1_selection_df': modality_PL_full, 'kwargs':{'l1_strength': 0.1}},
    2: {'l1_selection_df': modality_LN_full, 'kwargs':{'l1_strength': 0.1}},
    3: {'l1_selection_df': modality_LU_full, 'kwargs':{'l1_strength': 0.1}},
}

dfs_rad_filters = {
    0: {'l1_selection_df': modality_PC_full, 'kwargs':{'l1_strength': 0.1}},
    1: {'l1_selection_df': modality_PL_full, 'kwargs':{'l1_strength': 0.1}},
    2: {'l1_selection_df': modality_LN_full, 'kwargs':{'l1_strength': 0.1}},
}

In [12]:
def run_permutation(i):    
    
    print ("Running:", i+1)
    
    try:
        df_outcomes = df_outcomes_PRESHUFFLE.copy(deep=True)
        np.random.default_rng(i).shuffle(df_outcomes['label'])

        summary_dfs = {}
        summary_coefs = {}
        

        summary_dfs['LR Clinical'], summary_coefs['LR Clinical'] = train_LR(modality_dict['cnl_dem_labs'].dropna(), df_outcomes.loc[modality_dict['cnl_dem_labs'].dropna().index])
        summary_dfs['LR Clinical-Albumin'], summary_coefs['LR Clinical-Albumin'] = train_LR(modality_dict['cnl_dem_labs'].dropna()[['albumin']], df_outcomes.loc[modality_dict['cnl_dem_labs'].dropna().index])

        summary_dfs['LR Rad-PC'], summary_coefs['LR Rad-PC'] = train_LR(modality_dict['rad_lesion_pc'].dropna(), df_outcomes.loc[modality_dict['rad_lesion_pc'].dropna().index], dfs_rad_filters[0])
        summary_dfs['LR Rad-PL'], summary_coefs['LR Rad-PL'] = train_LR(modality_dict['rad_lesion_pl'].dropna(), df_outcomes.loc[modality_dict['rad_lesion_pl'].dropna().index], dfs_rad_filters[1])
        summary_dfs['LR Rad-LN'], summary_coefs['LR Rad-LN'] = train_LR(modality_dict['rad_lesion_ln'].dropna(), df_outcomes.loc[modality_dict['rad_lesion_ln'].dropna().index], dfs_rad_filters[2])

        summary_dfs['LR Rad-LU'], _       = train_LR(modality_dict['rad_lesion_lu'].dropna(), df_outcomes.loc[modality_dict['rad_lesion_lu'].dropna().index], dfs_rad_filters_lu[3])
        summary_dfs['LR Rad-Average']     = average_models(summary_dfs,['LR Rad-PC', 'LR Rad-PL', 'LR Rad-LN'])

        summary_dfs['LR IHC-G'], summary_coefs['LR IHC-G']        = train_LR(df_glcm, df_outcomes.loc[df_glcm.index], None)
        summary_dfs['LR IHC-A'], summary_coefs['LR IHC-A']        = train_LR(df_texture, df_outcomes.loc[df_texture.index], None)
        summary_dfs['LR IHC-S'], summary_coefs['LR IHC-S']        = train_LR(df_texture.loc[:, df_texture.columns.str.contains('skewness')], df_outcomes.loc[df_texture.index], None)
        summary_dfs['LR PDL1-TPS'], _     = train_LR(df_pdl1, df_outcomes.loc[df_pdl1.index], None)

        summary_dfs['LR Gen-Only-TMB'], summary_coefs['LR Gen-Only-TMB'] = train_LR(df_tmb, df_outcomes.loc[df_tmb.index], None)
        summary_dfs['LR Gen-No-TMB'], summary_coefs['LR Gen-No-TMB']   = train_LR(df_nontmb, df_outcomes.loc[df_tmb.index], None)
        summary_dfs['LR Gen-Combined'], summary_coefs['LR Gen-Combined'] = train_LR(df_genomic, df_outcomes.loc[df_genomic.index], None)
        summary_dfs['LR Gen-Average']  = average_models(summary_dfs,['LR Gen-Only-TMB', 'LR Gen-No-TMB'])

        summary_dfs['LR Path-A-Average'] = average_models(summary_dfs,['LR IHC-A', 'LR PDL1-TPS'])
        summary_dfs['LR Path-G-Average'] = average_models(summary_dfs,['LR IHC-G', 'LR PDL1-TPS'])


        data, mask, labels = get_training_data(['gen_driver_tmb'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM TMB'], _ = train(data, mask, labels, {}, model_params_gate_off)

        data, mask, labels = get_training_data(['cnl_pdl1_score'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM PDL1'], _ = train(data, mask, labels, {}, model_params_gate_off)

        data, mask, labels = get_training_data(['gen_driver_tmb', 'cnl_pdl1_score'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM TMB+PDL1'], _ = train(data, mask, labels, {}, model_params_gate_off)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad'], _ = train(data, mask, labels, dfs_rad_filters, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'rad_lesion_lu'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad-LU'], _ = train(data, mask, labels, dfs_rad_filters_lu, model_params)

        data, mask, labels = get_training_data(['path_ihc_pdl1'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM IHC-A'], _ = train(data, mask, labels, {}, model_params_gate_off)

        data, mask, labels = get_training_data(['gen_driver_mut_amp'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Gen'], _ = train(data, mask, labels, {}, model_params_gate_off)

        data, mask, labels = get_training_data(['cnl_pdl1_score', 'gen_driver_mut_amp'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM TMB+PDL1'], _ = train(data, mask, labels, {}, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'path_ihc_pdl1'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad+IHC-A'], _ = train(data, mask, labels, dfs_rad_filters, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'path_ihc_glcm'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad+IHC-G'], _ = train(data, mask, labels, dfs_rad_filters, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'gen_driver_mut_amp'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad+Gen'], _ = train(data, mask, labels, dfs_rad_filters, model_params)

        data, mask, labels = get_training_data(['path_ihc_pdl1', 'gen_driver_mut_amp'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM IHC-A+Gen'], _ = train(data, mask, labels, {}, model_params)

        data, mask, labels = get_training_data(['path_ihc_glcm', 'gen_driver_mut_amp'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM IHC-G+Gen'], _ = train(data, mask, labels, {}, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'path_ihc_pdl1', 'gen_driver_mut_amp'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad+IHC-A+Gen'], summary_coefs['DyAM Rad+IHC-A+Gen'] = train(data, mask, labels, dfs_rad_filters, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'path_ihc_glcm', 'gen_driver_mut_amp'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad+IHC-G+Gen'], summary_coefs['DyAM Rad+IHC-G+Gen'] = train(data, mask, labels, dfs_rad_filters, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'path_ihc_pdl1', 'gen_driver_non_tmb', 'cnl_pdl1_score'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad+IHC-A+MutAmp'], _ = train(data, mask, labels, dfs_rad_filters, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'path_ihc_pdl1', 'gen_driver_mut_amp', 'cnl_pdl1_score'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad+IHC-A+Gen+PDL1'], summary_coefs['DyAM Rad+IHC-A+Gen+PDL1']  = train(data, mask, labels, dfs_rad_filters, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'path_ihc_glcm', 'gen_driver_mut_amp', 'cnl_pdl1_score'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad+IHC-G+Gen+PDL1'], summary_coefs['DyAM Rad+IHC-G+Gen+PDL1']  = train(data, mask, labels, dfs_rad_filters, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'path_ihc_pdl1', 'gen_driver_non_tmb', 'gen_driver_tmb', 'cnl_pdl1_score'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad+IHC-A+Gen+TMB+PDL1'], _ = train(data, mask, labels, dfs_rad_filters, model_params)

        data, mask, labels = get_training_data(['rad_lesion_pc', 'rad_lesion_pl', 'rad_lesion_ln', 'path_ihc_pdl1', 'gen_driver_mut_amp', 'cnl_pdl1_score', 'cnl_dem_labs'], modality_dict, modality_MASK, df_outcomes)
        summary_dfs['DyAM Rad+IHC-A+Gen+PDL1+Labs'], _ = train(data, mask, labels, dfs_rad_filters, model_params)

        summary_dfs['LR Multimodal-Average'] = average_models(summary_dfs,
         ['LR Rad-PC', 'LR Rad-PL', 'LR Rad-LN', 'LR IHC-A', 'LR PDL1-TPS', 'LR Gen-Combined']
        )

        
        summary_aucs = []

        for model in summary_dfs.keys():
            auc, ci = auc_roc_ci((summary_dfs[model]['label']), summary_dfs[model]['score'], 0.68)
            error = (ci[1] - ci[0]) / 2.0

            summary_aucs.append({'model':model, 'auc': auc, 'error': error, 'fold': i + 1})

        pd.DataFrame(summary_aucs).to_csv(f'tests/null_permutation/perm_{i+1}_aucs_all_1sigma.csv', index=False)
        return i
    except Exception as e:
        return str(e)
   

In [13]:
from dask.distributed import Client
from dask.distributed import wait, as_completed

if not 'client' in locals():
    client = Client(threads_per_worker=1, host='pllimsksparky3', n_workers=22, memory_limit='150GB')
client

Client Scheduler: tcp://10.254.130.16:17820 Dashboard: http://10.254.130.16:8787/status,Cluster Workers: 22 Cores: 22 Memory: 3.30 TB


In [14]:
futures = []
for i in range(0, 22):
    future = client.submit(run_permutation, i)
    futures.append(future)


In [15]:
for job in as_completed(futures):
    print (job.result())

This solver needs samples of at least 2 classes in the data, but the data contains only one class: 1
Found array with 0 feature(s) (shape=(168, 0)) while a minimum of 1 is required by RobustScaler.
1
2
5
3
0
9
16
12
19
14
8
10
6
13
15
17
21
20
18
11


In [19]:
milr_model_params = {'steps':250, 'hidden_size':32, 'class_weight': 'balanced', 'lr': 0.005, 'alpha': 0.005}


def run_permutation_milr(i):    
    
    print ("Running:", i+1)
    
    try:
        df_outcomes = df_outcomes_PRESHUFFLE.copy(deep=True)
        np.random.default_rng(i).shuffle(df_outcomes['label'])

        summary_dfs = {}
        summary_coefs = {}
        
        df_lesions_data, df_lesions_outcomes = get_lesion_dfs(
            df_radiology_by_site, 
            df_outcomes,
            index_col='main_index')

        summary_dfs['MILR Rad-Lesions'] = train_MILR(
            df_lesions_data.drop(columns=['job_tag', 'global_lesion_id', 'site'], errors='ignore'), 
            df_outcomes.loc[df_outcomes.index.isin(df_lesions_data.index)], 
            None, n_splits=10, 
            model_params=milr_model_params)
        
        summary_aucs = []

        for model in summary_dfs.keys():
            auc, ci = auc_roc_ci((summary_dfs[model]['label']), summary_dfs[model]['score'], 0.68)
            error = (ci[1] - ci[0]) / 2.0

            summary_aucs.append({'model':model, 'auc': auc, 'error': error, 'fold': i + 1})

        pd.DataFrame(summary_aucs).to_csv(f'tests/null_permutation_MILR/perm_{i+1}_aucs_all_1sigma.csv', index=False)
        return i
    except Exception as e:
        return str(e)


In [20]:
for i in range(0, 20):
    run_permutation_milr(i)

Running: 1
100%|██████████| 10/10 [13:36<00:00, 81.69s/it]
Agg [10] AUC = 0.512 +/- 0.097 95% CL
Running: 2
100%|██████████| 10/10 [11:01<00:00, 66.14s/it]
Agg [10] AUC = 0.522 +/- 0.096 95% CL
Running: 3
100%|██████████| 10/10 [08:39<00:00, 51.91s/it]
Agg [10] AUC = 0.598 +/- 0.092 95% CL
Running: 4
100%|██████████| 10/10 [08:31<00:00, 51.14s/it]
Agg [10] AUC = 0.450 +/- 0.093 95% CL
Running: 5
100%|██████████| 10/10 [08:31<00:00, 51.13s/it]
Agg [10] AUC = 0.614 +/- 0.098 95% CL
Running: 6
100%|██████████| 10/10 [08:31<00:00, 51.14s/it]
Agg [10] AUC = 0.505 +/- 0.103 95% CL
Running: 7
100%|██████████| 10/10 [09:07<00:00, 54.71s/it]
Agg [10] AUC = 0.502 +/- 0.099 95% CL
Running: 8
100%|██████████| 10/10 [08:40<00:00, 52.08s/it]
Agg [10] AUC = 0.408 +/- 0.091 95% CL
Running: 9
100%|██████████| 10/10 [08:33<00:00, 51.39s/it]
Agg [10] AUC = 0.476 +/- 0.097 95% CL
Running: 10
100%|██████████| 10/10 [08:35<00:00, 51.50s/it]
Agg [10] AUC = 0.433 +/- 0.100 95% CL
Running: 11
100%|██████████| 